# O005-BRIDGE-C1 — Alur Kerja Python/Jupyter yang Reprodusibel

Notebook ini adalah tambahan independen untuk edisi Bahasa Indonesia *Introduction to Mathematical Modeling*. Ia memperagakan rantai bukti komputasional dengan model pendinginan sederhana. Data dibuat secara sintetis dan tidak merupakan bukti eksperimen.

Produksi dan QA: OpenAI Codex gpt-5.6-sol, Ultra. Kredit Joceline Lega dan University of Arizona untuk buku sumber tetap terpisah dan tidak tersirat sebagai dukungan terhadap tambahan ini. Distribusi mengikuti CC BY-NC-SA 4.0.


In [ ]:
import hashlib
import json
import platform

import matplotlib
import matplotlib.pyplot as plt
import numpy as np
import scipy
from scipy.optimize import curve_fit

CANONICAL_SEED = 20260822
SEED = CANONICAL_SEED
CANONICAL_DATA_SHA256 = "1000bc1092f173258d2be37e4f8906ea0933708582d09768ce96eed739be337e"
EXPECTED = {
    "python": "3.13.9",
    "numpy": "2.4.4",
    "scipy": "1.17.1",
    "matplotlib": "3.10.9",
}
VERSIONS = {
    "python": platform.python_version(),
    "numpy": np.__version__,
    "scipy": scipy.__version__,
    "matplotlib": matplotlib.__version__,
}

def require(condition, message):
    if not bool(condition):
        raise RuntimeError(message)

require(VERSIONS == EXPECTED, f"Lingkungan berbeda: {VERSIONS!r} != {EXPECTED!r}")


## Kontrak model

Dengan waktu $t$ dalam menit dan suhu dalam derajat Celsius,

$$T(t)=T_{\infty}+(T_0-T_{\infty})e^{-kt}.$$

Notebook menetapkan $T_0=92$, mensyaratkan $k>0$, dan mengestimasi $k$ serta $T_{\infty}$. Fungsi model tidak membaca keadaan global yang tersembunyi: setiap masukan yang berubah dicantumkan sebagai argumen atau konstanta bernama.


In [ ]:
T0_C = 92.0
TRUE_K_PER_MIN = 0.073
TRUE_T_INF_C = 22.4
NOISE_SD_C = 0.35

def suhu_pendinginan(waktu_menit, laju_per_menit, suhu_lingkungan_c, suhu_awal_c=T0_C):
    waktu = np.asarray(waktu_menit, dtype=float)
    return suhu_lingkungan_c + (suhu_awal_c - suhu_lingkungan_c) * np.exp(-laju_per_menit * waktu)

def serialisasi_csv(waktu_menit, suhu_c):
    baris = ["time_min,temperature_c"]
    baris.extend(f"{t:.1f},{y:.6f}" for t, y in zip(waktu_menit, suhu_c))
    return "\n".join(baris) + "\n"

require(np.isclose(suhu_pendinginan(0.0, TRUE_K_PER_MIN, TRUE_T_INF_C), T0_C), "Kondisi awal model gagal")
require(suhu_pendinginan(50.0, TRUE_K_PER_MIN, TRUE_T_INF_C) > TRUE_T_INF_C, "Solusi pendinginan melewati suhu lingkungan")


## Data sintetis dan asal-usulnya

Sebelas waktu pengamatan ditentukan terlebih dahulu. Pembangkit acak lokal memakai benih eksplisit. Aturan serialisasi—nama kolom, satu angka desimal untuk waktu, enam angka desimal untuk suhu, UTF-8, dan satu baris akhir—merupakan bagian dari identitas data.


In [ ]:
def buat_data(seed=SEED):
    rng = np.random.default_rng(seed)
    waktu = np.arange(0.0, 55.0, 5.0)
    suhu_bersih = suhu_pendinginan(waktu, TRUE_K_PER_MIN, TRUE_T_INF_C)
    # The serialized values are also the values used for estimation, so the
    # published SHA-256 binds the exact analyzed data rather than a hidden
    # higher-precision array.
    suhu_amatan = np.round(suhu_bersih + rng.normal(0.0, NOISE_SD_C, waktu.size), 6)
    csv_text = serialisasi_csv(waktu, suhu_amatan)
    data_sha256 = hashlib.sha256(csv_text.encode("utf-8")).hexdigest()
    return waktu, suhu_amatan, csv_text, data_sha256

waktu_menit, suhu_amatan_c, csv_text, DATA_SHA256 = buat_data()
require(waktu_menit.size == suhu_amatan_c.size == 11, "Sensus data sintetis berbeda")
require(len(csv_text.encode("utf-8")) == 186, "Ukuran CSV sintetis berbeda")
if SEED == CANONICAL_SEED:
    require(DATA_SHA256 == CANONICAL_DATA_SHA256, "Hash data kanonik berbeda")
else:
    require(DATA_SHA256 != CANONICAL_DATA_SHA256, "Benih nonkanonik tidak mengubah identitas data")
    print(f"Benih nonkanonik {SEED}; identitas data berubah menjadi {DATA_SHA256}.")


## Pisahkan dahulu, lalu estimasi

Indeks 0–7 menjadi data latih. Tiga pengamatan terakhir disisihkan sebagai data uji dan tidak ikut menentukan parameter. Pemisahan berurutan ini menguji prediksi ke waktu yang lebih lanjut; ia bukan satu-satunya desain validasi yang mungkin.


In [ ]:
INDEKS_LATIH = np.arange(0, 8)
INDEKS_UJI = np.arange(8, 11)

def estimasi_parameter(waktu, suhu):
    parameter, kovariansi = curve_fit(
        lambda t, k, t_inf: suhu_pendinginan(t, k, t_inf),
        waktu[INDEKS_LATIH],
        suhu[INDEKS_LATIH],
        p0=(0.05, 20.0),
        bounds=([0.001, 0.0], [0.5, 50.0]),
        maxfev=10000,
    )
    return parameter, kovariansi

parameter_hat, kovariansi_hat = estimasi_parameter(waktu_menit, suhu_amatan_c)
k_hat, t_inf_hat = parameter_hat
prediksi_c = suhu_pendinginan(waktu_menit, k_hat, t_inf_hat)
residu_latih_c = suhu_amatan_c[INDEKS_LATIH] - prediksi_c[INDEKS_LATIH]


## Pemeriksaan numerik dan visual

RMSE data latih memadatkan besar residu, korelasi residu–waktu memeriksa satu pola sederhana, dan MAE data uji yang disisihkan (*holdout*) mengukur kesalahan pada titik yang tidak dipakai dalam estimasi. Ketiganya harus dibaca bersama grafik dan kontrak model; tidak satu pun merupakan uji kecukupan universal.

**Deskripsi panjang gambar:** panel kiri memperlihatkan delapan titik data latih, tiga titik data uji pada waktu paling akhir, dan kurva pendinginan terestimasi yang menurun menuju suhu lingkungan. Panel kanan memperlihatkan residu data latih terhadap waktu di sekitar garis nol; tidak tampak tren satu arah yang kuat pada realisasi sintetis ini.


In [ ]:
RMSE_LATIH_C = float(np.sqrt(np.mean(residu_latih_c**2)))
MEAN_RESIDU_C = float(np.mean(residu_latih_c))
KORELASI_RESIDU_WAKTU = float(np.corrcoef(waktu_menit[INDEKS_LATIH], residu_latih_c)[0, 1])
MAE_UJI_C = float(np.mean(np.abs(suhu_amatan_c[INDEKS_UJI] - prediksi_c[INDEKS_UJI])))

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(9, 3.5), constrained_layout=True)
ax1.scatter(waktu_menit[INDEKS_LATIH], suhu_amatan_c[INDEKS_LATIH], marker="o", label="data latih")
ax1.scatter(waktu_menit[INDEKS_UJI], suhu_amatan_c[INDEKS_UJI], marker="s", label="data uji")
ax1.plot(waktu_menit, prediksi_c, color="black", label="model terestimasi")
ax1.set(xlabel="waktu (menit)", ylabel="suhu (°C)")
ax1.legend()
ax2.axhline(0.0, color="black", linewidth=1)
ax2.scatter(waktu_menit[INDEKS_LATIH], residu_latih_c)
ax2.set(xlabel="waktu (menit)", ylabel="residu data latih (°C)")
fig.canvas.draw()
if matplotlib.get_backend().lower() != "agg":
    plt.show()
plt.close(fig)

require(np.isfinite([RMSE_LATIH_C, MEAN_RESIDU_C, KORELASI_RESIDU_WAKTU, MAE_UJI_C]).all(), "Diagnostik mengandung nilai tak hingga atau NaN")


## Ringkasan yang dapat diperiksa

Ringkasan berikut menggunakan nama dan pembulatan tetap. Pemeriksaan numerik memakai nilai tak dibulatkan. Toleransi sengaja lebih longgar daripada digit terakhir pecahan mengambang, tetapi cukup ketat untuk menangkap perubahan data, model, pembagian, atau algoritma.


In [ ]:
RINGKASAN = {
    "unit_id": "O005-BRIDGE-C1",
    "seed": SEED,
    "data_sha256": DATA_SHA256,
    "k_hat_per_min": round(float(k_hat), 8),
    "t_inf_hat_c": round(float(t_inf_hat), 8),
    "rmse_latih_c": round(RMSE_LATIH_C, 8),
    "mean_residu_c": round(MEAN_RESIDU_C, 8),
    "korelasi_residu_waktu": round(KORELASI_RESIDU_WAKTU, 8),
    "mae_uji_c": round(MAE_UJI_C, 8),
    "versions": VERSIONS,
}

if SEED == CANONICAL_SEED:
    require(abs(k_hat - TRUE_K_PER_MIN) < 0.003, "Estimasi k keluar dari batas")
    require(abs(t_inf_hat - TRUE_T_INF_C) < 0.8, "Estimasi T_inf keluar dari batas")
    require(RMSE_LATIH_C < 0.5, "RMSE data latih keluar dari batas")
    require(abs(MEAN_RESIDU_C) < 0.1, "Rerata residu keluar dari batas")
    require(abs(KORELASI_RESIDU_WAKTU) < 0.3, "Korelasi residu-waktu keluar dari batas")
    require(MAE_UJI_C < 0.4, "MAE data uji keluar dari batas")
    require(RINGKASAN["k_hat_per_min"] == 0.07287235, "Nilai kanonik k berubah")
    require(RINGKASAN["t_inf_hat_c"] == 22.63964536, "Nilai kanonik T_inf berubah")
    require(RINGKASAN["rmse_latih_c"] == 0.31976406, "Nilai kanonik RMSE berubah")
    require(RINGKASAN["mae_uji_c"] == 0.17952331, "Nilai kanonik MAE berubah")
print(json.dumps(RINGKASAN, ensure_ascii=False, sort_keys=True, indent=2))


## Verifikasi ulang deterministik dalam kernel yang sama

Sel terakhir mengulang pembuatan data dan estimasi melalui fungsi yang sama, lalu membandingkan byte data dan parameter dengan hasil pertama. Pemeriksaan ini mendeteksi ketidakdeterministikan dalam satu kernel, tetapi bukan pengganti gerbang QA eksternal yang memulai kernel Jupyter baru dan menjalankan semua sel secara berurutan.


In [ ]:
waktu_ulang, suhu_ulang, csv_ulang, hash_ulang = buat_data(SEED)
parameter_ulang, _ = estimasi_parameter(waktu_ulang, suhu_ulang)

require(csv_ulang.encode("utf-8") == csv_text.encode("utf-8"), "Byte CSV berubah dalam kernel yang sama")
require(hash_ulang == DATA_SHA256, "Hash data berubah dalam kernel yang sama")
require(np.array_equal(waktu_ulang, waktu_menit), "Grid waktu berubah dalam kernel yang sama")
require(np.array_equal(suhu_ulang, suhu_amatan_c), "Data sintetis berubah dalam kernel yang sama")
require(np.allclose(parameter_ulang, parameter_hat, rtol=0.0, atol=1e-12), "Estimasi berubah dalam kernel yang sama")
print("Verifikasi ulang deterministik dalam kernel yang sama lulus.")
